# Submit a Parameter Ensemble

To recap what we've done so far, we created a set of parameter files with `meristem.paramgen`: 6 FATES parameter files 
(`fates_oat_0000.json`...`fates_oat_0005.json`). Each one sets one of 3 parameters to its minimum or maximum value, as defined in `meristem/examples/paramgen/fates_example/fates_param_dir/main.csv`.

Next, we will use `meristem.ensemble` to run a CLM-FATES case with each of those parameter files, and to track and manage the whole ensemble. 

## meristem.ensemble

`meristem.ensemble` handles creation, submission, and tracking/management of parameter ensembles of CLM and CLM-FATES. Each ensemble member is a clone of one already-built case. `meristem` sets each member's parameter file in the clone and submits it, so nothing needs to be rebuilt. 

<div class="alert alert-warning">
  <strong>Important Note:</strong> All of the command-line steps in this notebook should be done on Derecho, not from Casper. The Python cells that write the script and YAML can still run in Jupyter on Casper, since they share the same filesystem.
<br>
<br>
You should still make sure that the kernel for this notebook is "meristem" (see upper right corner, above).
</div>

### Requirements

To run `meristem.ensemble` you need:

- A **built base case**: for us, `BONA.FATESsp` (see below)
- **Paramgen output**: for us, `/glade/u/home/$USER/paramfiles_OAT`
- `meristem` environment: for us, the `meristem` conda env from the [overview notebook](01_sensitivity_overview.ipynb)

## Step 1: Check (or generate) the base case

As our base case, we will use the `BONA.FATESsp` case from the [single-point exercises](../singlepoint/exercises/FATESsp_NEON.ipynb). Navigate to that directory:

```bash
cd /glade/u/home/$USER/cases/BONA.FATESsp
```

In order for `meristem.ensemble` to clone this case (`./create_clone` from `cime/scripts`), the case must be built. Check that the build is complete:

```bash
./xmlquery BUILD_COMPLETE
```

This should give `TRUE`. If not, run `qcmd -- ./case.build` inside the case directory. Or, if you skipped this exercise, complete it now.

We will also use this base case as our "default" run for comparison against our ensemble of parameter-modified runs. So if you haven't run this case yet, be sure to do that as well (`./case.submit`).

Together with our 6 members it makes up the 2N+1 runs of a one-at-a-time experiment. The members start in 2014 like the base case, so we'll compare them over 2014–2015.

---

Because we are cloning this case for each of our ensemble members, each of our clones will inherit the FATES-SP settings, the `hist_fincl2` we added to the `user_nl_clm`, and all of the `xmlchange` operations we completed in the exercise.


<div class="alert alert-warning">
  <strong>Important Note:</strong> meristem doesn't create the member cases immediately upon <code>ensemble create</code> or <code>ensemble submit</code>. This happens in several member-specific setup jobs. So if you modify any xml or namelist variables in the base case before the setup jobs finish, they will carry over to your member clones.
  
  Be sure to make any edits to the base case <em>after</em> the setup jobs have finished. Or, for safety, don't make any edits to the base case once you have initialized your ensemble.
</div>

## Step 2: Write a stage script

Sometimes, we want to make modifications to our clones from the base case. `meristem.ensemble` facilitates this through running an optional user-supplied script during the case creation phase of each member case. After each clone is created and set up (`./case.setup`), `meristem` runs the user-supplied script. Then `meristem` sets the member's parameter file and submits the case.

This script is thus the place for any `xmlchange` or `user_nl_clm` modifications, but **should not** set the `paramfile` or `fates_paramfile`, nor should it run `./case.submit`. `meristem` handles both of these operations.

The script is called from inside each members `root` directory, and is called with `bash <script> <case_root> <previous_case_name> <previous_case_root>`.

The `<previous_case_name>` and `<previous_case_root>` are used in "tethering" different model stages together ([see the meristem run documentation](https://github.com/adrifoster/meristem/blob/main/docs/run.md)), but we will not make use of that feature here.

Because the script is called from each member's `root` directory (not its case directory) the script should first `cd` into the case directory with `cd $1`.

The below code creates a small script that `meristem` will run once for each member case. Run it here to create one we can use.

Be sure to change the `user` variable to your username on the NCAR machines.

In [ ]:
import os
user = os.getenv('USER')
print(f"username is {user}, update to your username if this is not correct")

In [ ]:
from pathlib import Path

script = """\
#!/usr/bin/env bash

if [ $# -lt 1 ]; then
  echo "ERROR: please specify a case_root"
  exit 1
fi
case_root="$1"

cd ${case_root}

./xmlchange --subgroup case.run JOB_WALLCLOCK_TIME=01:00:00
./xmlchange --subgroup case.st_archive JOB_WALLCLOCK_TIME=01:00:00
./xmlchange STOP_OPTION="nyears"
./xmlchange STOP_N=2
./xmlchange RESUBMIT=0
./xmlchange CLM_FORCE_COLDSTART=on
"""

script_path = Path(f"/glade/u/home/{user}/code/setup_oat.sh")
script_path.write_text(script)
print(f"Wrote {script_path}")

Notice that we:

1. Used the first argument to the script (`$1`, the case directory) to change into the case directory *before* making any xml modifications. If we had skipped that step, our script would have encountered a runtime error.
2. **Did not** call `case.submit`. This is crucial, as `meristem` handles this part.


`meristem` runs the script in "strict mode" (`set -euo pipefail`). If any command in it fails, such as forgetting to `cd` into the case directory first, or a typo in an `xmlchange` variable name, the script stops and that member's stage is marked as failed, rather than the case being submitted with only some of your changes applied. We'll see how to find and retry failed members later in this notebook.

<div class="alert alert-info">
<strong>Exercise: Add output to every member</strong><br><br>

Suppose we also want every ensemble member to write <code>FATES_NPP_PF</code> and <code>FATES_GPP_PF</code> to its primary history file (h0).

How would you change the script above so that each member's <code>user_nl_clm</code> requests these variables?

<em>Hint:</em> the script runs inside each member's case directory (after <code>cd ${case_root}</code>), so anything you could do by hand in a case directory, you can do here.
</div>

<div class="alert alert-success">
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

Add a line to the end of the script that appends to the clone's <code>user_nl_clm</code>:

```bash
echo "hist_fincl1 = 'FATES_NPP_PF','FATES_GPP_PF'" >> user_nl_clm
```

Because the script runs on each clone, this changes only the members' <code>user_nl_clm</code> files; <code>BONA.FATESsp</code> itself is left untouched. <code>meristem</code> then adds the member's <code>fates_paramfile</code> afterwards.

Note: each clone starts with a copy of the base case's <code>user_nl_clm</code>. If <code>hist_fincl1</code> were already set there, a second <code>hist_fincl1 = ...</code> would silently replace it, while <code>+=</code> adds variables to the existing list.

</details>
</div>

## Step 3: Write an ensemble YAML file

`meristem.ensemble` reads what it needs to know about an ensemble from a YAML configuration file. This file tells `meristem`:

- **what to call the ensemble** and **where to keep track of it** (`ensemble_id`, `root`)
- **which case to clone** for each member (`base_case`)
- **where to find the parameter files**, and which `user_nl_clm` variable to point at them (`paramgen`, `paramfile_key`)
- **what each member should run** (`member_stages`)

Once the ensemble is created, `meristem` saves a copy of this configuration in the ensemble's `root` directory, so the YAML file is only needed once.


Let's look at each part:

| Key | What it does | Our value |
|---|---|---|
| `ensemble_id` | A short name for the ensemble. It is used in job names and in every member's case name. | `oat` |
| `root` | Where `meristem` keeps the ensemble's state files, job scripts, and logs. Member cases are created in `<root>/cases`. | `/glade/u/home/$USER/cases/oat_ensemble` |
| `output_root` | CIME output root for the clones. | `/glade/derecho/scratch/$USER/oat_ensemble` |
| `project` | The project code to charge the jobs to. | your project code |
| `base_case` | The built case every member is cloned from. | `BONA.FATESsp` |
| `paramfile_key` | The `user_nl_clm` variable `meristem` sets to each member's parameter file: `paramfile` for CLM parameter files, `fates_paramfile` for FATES parameter files. | `fates_paramfile` |
| `paramgen` | Where the `paramgen` output lives (`dir`), the prefix of its files (`prefix`), and which members to run (`members`). | `fates_oat`, all members |
| `member_stages` | The stages each member runs, in order. Each one has a `name`, a PBS `queue` for its setup job, and an optional `script` (the one we just wrote). | one stage, `run` |

Each member's case will be named `<base case>_<ensemble_id>_<member>_<stage>`, so the first member's case will be `BONA.FATESsp_oat_fates_oat_0000_run`.

The cell below writes this file for us. Be sure to replace `XXXXXXXX` with your project code (UCGD0011 for this tutorial).

In [ ]:
config = f"""\
ensemble_id: oat
root: /glade/u/home/{user}/cases/oat_ensemble
output_root: /glade/derecho/scratch/{user}/oat_ensemble
project: PXXXXXXX

base_case: /glade/u/home/{user}/cases/BONA.FATESsp
paramfile_key: fates_paramfile

paramgen:
  dir: /glade/u/home/{user}/paramfiles_OAT
  prefix: fates_oat
  members: all

member_stages:
  - name: run
    script: /glade/u/home/{user}/code/setup_oat.sh
    queue: develop
"""
config_path = Path(f"/glade/u/home/{user}/ensemble.yaml")
config_path.write_text(config)
print(config_path.read_text(encoding='utf-8'))

## Step 4: Create the ensemble

Create the ensemble with:

```bash
cd /glade/u/home/$USER

conda activate meristem
meristem ensemble create --config ensemble.yaml
```

Run this in a Derecho terminal, like the rest of the command-line steps

This step submits nothing yet, and only creates the initial files and directories needed to set up the ensemble.

If it worked, you should see:

```
Initialized ensemble 'oat' at /glade/u/home/<user>/cases/oat_ensemble
Next step:  meristem ensemble submit --root /glade/u/home/<user>/cases/oat_ensemble
```

Before writing anything, `create` checks that:

- the base case exists and is built (`BUILD_COMPLETE` is `TRUE`)
- every member listed in `fates_oat.txt` has a parameter file in the supplied `paramgen` `dir`
- the stage script exists
- none of the member case directories already exist

If any check fails, `create` stops with an error explaining what is wrong, and nothing is written. Fix the problem and run the command again.

### What did `create` make?

Take a look inside the ensemble's `root`:

```bash
cd /glade/u/home/$USER/cases/oat_ensemble
ls
ls members
```

You should see:

- `ensemble_state.json`: the ensemble's state file. `meristem` uses it to keep track of every member, and it holds the copy of your configuration mentioned above.
- `members/`: one directory per member (`fates_oat_0000`...`fates_oat_0005`). Each has a `run_state.json` that tracks that member's stage. Later, these directories will also hold the member's PBS job scripts and logs.

Notice that there is no `cases/` directory yet. The member cases are cloned later, by each member's setup job, once we submit.

<div class="alert alert-warning">
<strong>Note:</strong> The state files and job scripts store absolute paths, so an ensemble can't be moved or renamed after it is created. Running <code>create</code> a second time with the same <code>root</code> also fails, because a state file already exists there. To start over, delete the ensemble with <code>rm -r /glade/u/home/$USER/cases/oat_ensemble</code> and run <code>create</code> again.
</div>

## Step 5: Test first

`meristem.ensemble` lets you first preview what would be generated or submitted by using the argument `--dry-run` for many commands. In the context of ensemble submission, `--dry-run` generates the PBS job scripts but does not submit them, so users can preview them if they want to.

Do that here:

```bash
meristem ensemble submit --root /glade/u/home/$USER/cases/oat_ensemble --dry-run
```

Nothing is submitted to the queue, and the ensemble's state doesn't change, so every member is still waiting to be submitted. You should see:

```
oat: dry run, wrote job scripts for 6 members in /glade/u/home/<user>/cases/oat_ensemble/members
```

### Look at a job script

Each member now has a PBS job script for its setup job. Look at the first member:

```bash
cd /glade/u/home/$USER/cases/oat_ensemble/members/fates_oat_0000
ls
```

You should see the run script (`oat_fates_oat_000_run.pbs`) and this run's model state (`run_state.json`).

Check out the job script:

```bash
cat oat_fates_oat_0000_run.pbs
```

The script is named `<ensemble_id>_<member>_<stage>.pbs`. In it, you'll see:

1. **`#PBS` lines**: the job's name, queue (`develop`, from our YAML), project, walltime, and log file.
2. **`meristem ensemble prepare`**: clones `BONA.FATESsp` into this member's case (with `--keepexe`, so nothing is rebuilt) and runs `case.setup`.
3. **Our stage script** (`setup_oat.sh`), run on the new clone.
4. **`meristem ensemble finalize`**: sets `fates_paramfile` in the clone's `user_nl_clm` to this member's parameter file and checks that the generated `lnd_in` points at it.
5. **`./case.submit`**: submits the member's CLM run.
6. **`meristem run record-cime-jobs`**: records job-ids of the submitted CIME jobs
7. **`meristem run submit-advance` and `submit-fail`**: small follow-up jobs that wait for the CLM run and record whether it finished or failed.

Because finalize runs *after* our script, nothing in the stage script can accidentally overwrite the member's parameter file.

<div class="alert alert-warning">

**Note:** The queue here is for the _setup job_, not the actual CLM-FATES case. To modify the job queues for your actual CLM jobs, use xmlchange commands inside the stage script.
```bash
./xmlchange JOB_QUEUE=tutorial
```
</div>

### Submit one member

`meristem.ensemble` also allows you to only submit a specified number of members with `--limit N`. This is useful if you want to test one run first before submitting the entire ensemble, or if you are running a large ensemble and are worried about running up against your cluster's job queue limits. Ensemble members are submitted in order, and calling `meristem ensemble submit` again submits only non-submitted members.

Submit just the first member, as a test:

```bash
meristem ensemble submit --root /glade/u/home/$USER/cases/oat_ensemble --limit 1
```

`--limit 1` submits one member (`fates_oat_0000`) and leaves the rest waiting. You should see:

```bash
oat: submitted 1 members
Submitted ensemble 'oat' at /glade/u/home/<user>/cases/oat_ensemble
```

Check that its setup job is in the queue (this may take a couple minutes to appear):

```bash
qstat -u $USER
```

You should see a job named `oat_fates_oat_0000_run`. When it finishes (a minute or two), its log is in the member's directory (`oat_fates_oat_0000_run.log`), and the member's case exists at `/glade/u/home/$USER/cases/oat_ensemble/cases/BONA.FATESsp_oat_fates_oat_0000_run`:

```bash
cd /glade/u/home/$USER/cases/oat_ensemble/members/fates_oat_0000
cat oat_fates_oat_0000_run.log
```

The last line of the log file should be something like:

```
=== Setup complete: Wed Nov  4 09:43:45 AM MST 2026 ===
```

<div class="alert alert-info">
<strong>Exercise: Did the member get the right parameter file?</strong><br><br>

Once the setup job for <code>fates_oat_0000</code> has finished, go into its case directory and check which parameter file it is using. Then use <code>fates_oat_key.csv</code> to find out which parameter this member changed, and in which direction.
</div>

<div class="alert alert-success">
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>

```bash
cd /glade/u/home/$USER/cases/oat_ensemble/cases/BONA.FATESsp_oat_fates_oat_0000_run
grep fates_paramfile user_nl_clm CaseDocs/lnd_in
```

Both should point to <code>/glade/u/home/$USER/paramfiles_OAT/fates_oat_0000.json</code>.

Now see what parameter that ensemble member is modifying (paste the below into a new Jupyter Notebook cell):

```python
import pandas as pd

oat_key = pd.read_csv(f'/glade/u/home/{user}/paramfiles_OAT/fates_oat_key.csv')
oat_key.head()
```

In [ ]:
# paste any python code here for the above solution


## Step 6: Submit the rest

Now that we have confirmed the jobs and cases look right, submit the rest of the ensemble:

```bash
meristem ensemble submit --root /glade/u/home/$USER/cases/oat_ensemble
```

Because `fates_oat_0000` was already submitted, this only submits the 5 members still waiting. You should see:

```
oat: submitted 5 members
Submitted ensemble 'oat' at /glade/u/home/<user>/cases/oat_ensemble
```

From here, each member runs on its own. Its setup job clones and sets up its case and submits the CLM run, and the follow-up jobs record when that run finishes. PBS dependencies chain these jobs together, so you don't need to do anything else. You can log out and come back later.

### Reading `qstat`

Check the queue again:

```bash
qstat -u $USER
```

You should see a mix of setup jobs (`oat_fates_oat_000N_run`), CLM jobs (`run.BONA.FATESsp_oat_...` and `st_archive...`), and the small follow-up jobs (`..._advance`, `..._fail`).

The `S` (state) column of `qstat -u $USER` shows where each job is:

| State | Meaning | What you might see in this ensemble |
|---|---|---|
| `Q` | **Queued**: waiting for resources to free up | Setup jobs and CLM runs waiting their turn |
| `R` | **Running** | A setup job cloning its case, or a CLM run in progress |
| `H` | **Held**: waiting on another job to finish first | The follow-up jobs (`..._advance`, `..._fail`), which wait for their member's CLM run, or an `st_archive` job |
| `E` | **Exiting**: finished, and PBS is cleaning up | Any job, briefly, right after it ends |

Seeing lots of `H` jobs is normal here. `meristem` chains each member's jobs together with PBS *dependencies*, so a job is "held" (`H`) until the job it depends on has finished, then moves to queued (`Q`) or running (`R`).

### How many jobs will I have?

Each member goes through a short chain of jobs:

1. `submit` queues one **setup job** for each member.
2. When a setup job runs, it submits that member's CLM jobs (`case.run`, and `case.st_archive` if archiving is on) plus two small follow-up jobs (`_advance` and `_fail`), then exits.

So each member has **up to 4 jobs** in the queue at once (assuming no `RESUBMIT`s), and most of them are held (`H`) waiting on `case.run`:

| Ensemble size | Right after `submit` | Peak, once every setup job has run |
|---|---|---|
| 6 (this tutorial) | 6 | ~24 |
| 100 | 100 | ~400 |
| 1000 | 1000 | ~4000 |


<div class="alert alert-warning">
<strong>Queue limits:</strong> Some HPC systems limit how many jobs one user can have queued or running, and the limits can differ per queue. Held jobs usually count toward them. A large ensemble submitted all at once can hit these limits, and then new jobs (including the ones a setup job tries to submit) are rejected and that member's stage fails.<br><br>

<code>meristem</code> does not manage these limits for you. For large ensembles, you can submit in batches with <code>--limit</code>, and run <code>submit</code> again as members finish. Each call only submits members that haven't started yet:

<pre>
meristem ensemble submit --root &lt;root&gt; --limit 500
# ...later, once some have finished
meristem ensemble submit --root &lt;root&gt; --limit 500
</pre>

<br>
Check your machine's documentation for its current limits.
</div>

## Step 7: Monitor

`qstat` only gives information about individual PBS jobs. To see where each *member* is, including ones that have finished, we will use `meristem ensemble status`:

```bash
meristem ensemble status --root /glade/u/home/$USER/cases/oat_ensemble
```

Partway through your ensemble run, the output will look something like this:

```text
Ensemble: oat (/glade/u/home/<user>/cases/oat_ensemble)
6 members

     pending  submitted  done  failed
run        0          4     2       0
```

Each member stage is in one of these states:

- **pending**: not submitted yet
- **submitted**: its jobs are queued or running (setup, CLM run, or follow-up)
- **done**: the CLM run finished successfully
- **failed**: something went wrong (see Step 8)

To list every member individually, add `--print-members`:

```bash
meristem ensemble status --root /glade/u/home/$USER/cases/oat_ensemble --print-members
```

will return

```
Ensemble: oat (/glade/u/home/<user>/cases/oat_ensemble)
6 members

     pending  submitted  done  failed
run        0          4     2       0

        member stage    status           job_id  attempts
fates_oat_0000   run      done 8068694.desched1         1
fates_oat_0001   run submitted 8068745.desched1         1
fates_oat_0002   run submitted 8068746.desched1         1
fates_oat_0003   run      done 8068747.desched1         1
fates_oat_0004   run submitted 8068748.desched1         1
fates_oat_0005   run submitted 8068749.desched1         1
```

### Write a manifest

We can also write a **manifest** for the ensemble: a table with one row per member that combines the paramgen key (which parameter each member changed, and in which direction) with where each member's case and output live.

```bash
meristem ensemble manifest --root /glade/u/home/$USER/cases/oat_ensemble -o /glade/u/home/$USER/oat_manifest.csv
```

Let's take a look at the manifest:

In [ ]:
manifest = pd.read_csv(f"/glade/u/home/{user}/oat_manifest.csv")
manifest.head()

## Step 8: When things go wrong

<div class="alert alert-warning">
<strong>Don't actually run these commands in this tutorial.</strong> Unless your runs failed, these won't work.
</div>

Sometimes a run fails. The setup job might hit an error, or the CLM run might crash. `meristem` records this, so `status` shows it:

```text
Ensemble: oat (/glade/u/home/<user>/cases/oat_ensemble)
6 members

     pending  submitted  done  failed
run        0          0     5       1

Failed (1): fates_oat_0003:run
  retry with: meristem ensemble retry --root /glade/u/home/<user>/cases/oat_ensemble --failed
```

### Find out what went wrong

Look at the logs to figure out why a member failed. You should look at these logs:

1. **The setup job's log**, in the member's directory. Errors from cloning, `case.setup`, your stage script, or setting the parameter file will be here.

2. **The CLM case's logs**, if the setup job finished but the model run failed. Go to the member's case and look at `CaseStatus`, then at the logs in its run directory:
   ```bash 
   cd /glade/u/home/$USER/cases/oat_ensemble/cases/BONA.FATESsp_oat_fates_oat_0003_run
   cat CaseStatus
   cd $(./xmlquery --value RUNDIR)
   ls -t *.log*
   ```

### Retry

Once you've fixed the problem, resubmit every failed member:

```bash
meristem ensemble retry --root /glade/u/home/$USER/cases/oat_ensemble --failed
```

To retry particular members instead, name them:

```bash
meristem ensemble retry --root /glade/u/home/$USER/cases/oat_ensemble --members fates_oat_0003
```

A retried member **reuses its existing case**, and by default the stage script runs on it again. Lines like `./xmlchange STOP_N=2` are safe to repeat, but lines that *append* to a file are not: `echo ... >> user_nl_clm` would add the same line a second time. If your script has already run successfully on the case, retry with `--skip-script`. This goes straight to setting the parameter file (safe to re-do) and resubmitting the case:

```bash
meristem ensemble retry --root /glade/u/home/$USER/cases/oat_ensemble --failed --skip-script
```

### Lost members

Sometimes, a member gets stuck as `submitted` even though none of its jobs are left in the queue. This can happen if, for example, the setup job ran out of walltime or a job was deleted with `qdel`. `meristem` refers to these members **lost**. Ask `meristem` to call out to PBS to check for these types of jobs with `--check-queue`:

```bash
meristem ensemble status --root /glade/u/home/$USER/cases/oat_ensemble --check-queue
```

If any were found, retry them with `--lost`:

```bash
meristem ensemble retry --root /glade/u/home/$USER/cases/oat_ensemble --lost
```

Before resubmitting, `retry --lost` checks PBS again, and it won't start a second copy of a member that still has a job queued or running.

### Cancel

To stop members, for example because you notice a mistake in your stage script after submitting, use `cancel`:

```bash
meristem ensemble cancel --root /glade/u/home/$USER/cases/oat_ensemble --dry-run   # see what would be cancelled
meristem ensemble cancel --root /glade/u/home/$USER/cases/oat_ensemble             # cancel everything
meristem ensemble cancel --root /glade/u/home/$USER/cases/oat_ensemble --members fates_oat_0004 # cancel just this member
```

`cancel` deletes the members' queued and running jobs (setup, CLM, and follow-up jobs) and marks them `failed`. Members that were already `done` are left alone. Use `cancel` rather than `qdel`: if you `qdel` jobs by hand, `meristem` doesn't know about it, and the members end up lost. After fixing the problem, `retry --failed` resubmits the cancelled members.

If you do `qdel` jobs directly, run `status` with `--check-queue` to mark them as lost and then you can retry as above.

## Step 9: Clean up

`meristem ensemble clean` deletes an ensemble's files. This is useful in two situations:

1. **When you no longer need the ensemble**, to delete the member cases and their run directories, which can take up a lot of space.
2. **When retrying isn't enough**, to wipe one or more members (or the whole ensemble) and start them over from scratch.

<div class="alert alert-warning">
<strong>Don't actually run these commands in this tutorial!</strong> We will use these members' output in the next notebook, so don't clean this ensemble yet.
</div>

In `meristem ensemble clean`, `--dry-run` lists every path `clean` would delete, without actually deleting anything:

```bash
meristem ensemble clean --root /glade/u/home/$USER/cases/oat_ensemble --dry-run
```

To wipe particular members, list them with `--members`. This deletes only their cases, run directories, and job scripts and logs. Their state is kept, so `status`, `manifest`, and `retry` still work:

```bash
meristem ensemble clean --root /glade/u/home/$USER/cases/oat_ensemble --members fates_oat_0003
meristem ensemble retry --root /glade/u/home/$USER/cases/oat_ensemble --members fates_oat_0003
```

Because the member's case is gone, `retry` clones a fresh one from `BONA.FATESsp`, just like the first time. This only works for members that have **failed**: `retry` won't rerun a member that's already `done`.

If any of those members still have jobs queued or running, `clean` refuses and deletes nothing. Add `--cancel` to cancel their jobs first, which also marks them `failed`.

### Delete the whole ensemble

With no `--members`, `clean` deletes everything: every member's case and run directory, and then the ensemble's `root`, including its state files, job scripts, and logs:

```bash
meristem ensemble clean --root /glade/u/home/$USER/cases/oat_ensemble
```

`clean` shows how many paths it will delete and asks you to confirm (supply `--yes` to bypass this confirmation check). Afterwards, you can run `meristem ensemble create` again with the same YAML file to start fresh.

What `clean` does not delete:

- any files doesn't know about (it will skip removing a directory if it isn't empty after deleting all the files it created) 
- the base case (`BONA.FATESsp`)
- the parameter files
- the build directory shared with the base case

By default, `clean` also skips each member's **archive** directories. Add `--include-archive` to delete those too.

## Next Steps

Next we will explore the parameter ensemble you just ran.

Click [here](04_ensemble_analysis.ipynb) to go to the next chapter.